# Length of Stay - Elixhauser

In [ ]:
import yaml
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import os

# Load all models
ALL_MODELS = yaml.safe_load(open("../config.yml"))["models"]

# Filter for Elixhauser LOS models
relevant_keys = [k for k in ALL_MODELS.keys() if k.startswith("elixhauser_") and "los" in k]
print(f"Models to train: {relevant_keys}")

## Index-Specific Train/Test Splits

Create views of the train/test splits which have the comorbidities for each index present.

### Data Aggregation

The training data can be quite large which results in extremely large inputs for the models. For example, the MACSS has 100 comorbidities so the training data is a N x 100 matrix where N is the number of rows.

To improve the training time for our models, we can 'compress' the data and represent it by identifying each unique combination of comorbidities and the number of times it occurred.

For example, given the following row-level data:

| COMORB_1 | COMORB_2 | COMORB_3 |
| -------- | -------- | -------- |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    1     |
|    1     |    1     |    0     |
|    1     |    0     |    1     |
|    1     |    0     |    1     |

We would represent it as:

| COMORB_1 | COMORB_2 | COMORB_3 | N |
| -------- | -------- | -------- | - |
|    1     |    1     |    1     | 7 |
|    1     |    1     |    0     | 1 |
|    1     |    0     |    1     | 2 |

In [ ]:
for model_key in relevant_keys:
    MODEL_CONFIG = ALL_MODELS[model_key]
    print(f"\nProcessing {model_key} (Target: {MODEL_CONFIG['class']})...")
    
    # Load Data
    elix_los_training = pd.read_csv(f"../datasets/{MODEL_CONFIG['dataset_training']}")
    elix_los_testing = pd.read_csv(f"../datasets/{MODEL_CONFIG['dataset_testing']}")
    
    # Data Aggregation
    elix_los_training_agg = elix_los_training.groupby(list(elix_los_training.columns),dropna=False).size().reset_index(name='N')
    elix_los_testing_agg = elix_los_testing.groupby(list(elix_los_testing.columns),dropna=False).size().reset_index(name='N')
    
    # Define input columns
    input_cols = [i for i in elix_los_training_agg.columns if "C_" in i]
    
    # Train
    clf = LinearRegression()
    clf.fit(elix_los_training_agg[input_cols], elix_los_training_agg[MODEL_CONFIG["class"]], sample_weight=elix_los_training_agg['N'])
    
    # Predict
    y_train_pred = clf.predict(elix_los_training_agg[input_cols])
    y_train_true = elix_los_training_agg[MODEL_CONFIG["class"]]
    
    y_test_pred = clf.predict(elix_los_testing_agg[input_cols])
    y_test_true = elix_los_testing_agg[MODEL_CONFIG["class"]]
    
    # Metrics
    mse_train = mean_squared_error(y_train_true, y_train_pred, sample_weight=elix_los_training_agg['N'])
    mae_train = mean_absolute_error(y_train_true, y_train_pred, sample_weight=elix_los_training_agg['N'])
    r2_train = r2_score(y_train_true, y_train_pred, sample_weight=elix_los_training_agg['N'])
    
    mse_test = mean_squared_error(y_test_true, y_test_pred, sample_weight=elix_los_testing_agg['N'])
    mae_test = mean_absolute_error(y_test_true, y_test_pred, sample_weight=elix_los_testing_agg['N'])
    r2_test = r2_score(y_test_true, y_test_pred, sample_weight=elix_los_testing_agg['N'])
    
    print(f"  Training R2: {r2_train:.4f}, MSE: {mse_train:.4f}")
    print(f"  Test R2: {r2_test:.4f}, MSE: {mse_test:.4f}")
    
    # Save
    if not os.path.exists('../models'):
        os.makedirs('../models')
    
    save_path = f'../models/los_elixhausers_{MODEL_CONFIG["class"]}.joblib'
    joblib.dump(clf, save_path)
    print(f"  Model saved to {save_path}")